# HAKE-MER — Step 0 baseline (GoEmotions protocol)

**Protocol:** batch 16, LR 5e-5, **4 epochs**, 3 seeds, best val F1-macro checkpoint.

## Phase 1 — DistilBERT only

1. **Runtime → Factory reset** → **GPU**
2. Run cells through **Download DistilBERT zip** (~1–1.5 h on T4)

## Phase 2 — RoBERTa (later)

1. **Uncomment** the RoBERTa training cell
2. Run it, then **Download RoBERTa zip**

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

### Phase 1 — train DistilBERT

In [ ]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}

In [ ]:
import json
from pathlib import Path

def show_campaign(name: str) -> None:
    path = Path("reference/artifacts") / name
    if not path.is_file():
        print(f"Missing {name}")
        return
    c = json.loads(path.read_text(encoding="utf-8"))
    print(path.name, "protocol:", c.get("protocol", {}))
    print("  epochs logged:", len(c["runs"][0]["history"]))
    for k, b in c["test_aggregate"].items():
        print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

show_campaign("baseline_plm_distilbert_base_uncased_campaign.json")

In [ ]:
import zipfile
from google.colab import files

def zip_backbone(slug: str, out_name: str) -> None:
    campaign = Path(f"reference/artifacts/baseline_plm_{slug}_campaign.json")
    if not campaign.is_file():
        raise FileNotFoundError(f"Run training first: {campaign}")
    zip_path = Path(f"/content/{out_name}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(campaign, campaign.name)
        for m in sorted(Path("runs").glob(f"{slug}_seed*_baseline_plm/metrics.json")):
            zf.write(m, f"{m.parent.name}/{m.name}")
    print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
    files.download(str(zip_path))

zip_backbone("distilbert_base_uncased", "baseline_plm_distilbert_step0.zip")

### Phase 2 — RoBERTa (uncomment the next cell, run it, then run the download cell below)

In [ ]:
# !./run_baseline_campaign.sh --backbone roberta-base {TRAIN_FLAGS}

In [ ]:
show_campaign("baseline_plm_roberta_base_campaign.json")

In [ ]:
zip_backbone("roberta_base", "baseline_plm_roberta_step0.zip")

## Integrity — save this Colab session

After each phase finishes, **File → Download → .ipynb** and commit the file under
`reference/training_records/step0_plm/colab/` (e.g. `20250926_distilbert_executed.ipynb`).
Keep cell outputs (git hash, training logs, `show_campaign`, zip download).
Artifacts go to `reference/artifacts/`; see `reference/training_records/README.md`.